# ABS Political

Economic metrics for each Australian Commonwealth Government, using ABS data.

## Python set-up

In [1]:
# system imports
import io
from pathlib import Path

# analytic imports
import mgplot as mg
import pandas as pd
import readabs as ra
from pandas import DataFrame, Series
from readabs import metacol as mc
from readabs.download_cache import get_file

# local imports
from abs_gdp import get_gdp
from abs_population import get_population
from abs_prices import (
    get_cpi,
    get_house_price_index,
    get_house_price_splice_report,
    seasonally_adjust,
)
from abs_spliced_series import (
    get_productivity_index,
    get_productivity_splice_report,
    get_unemployment_backcast_stats,
    get_unemployment_rate,
    get_unemployment_splice_report,
)
from decompose import FINAL_SEASADJ, FINAL_TREND, decompose

# this notebook's epoch helpers
from political import (
    cagr_by_government,
    cagr_path_by_government,
    change_by_government,
    continuous_index_by_government,
    cumulative_growth_by_government,
    epoch_vlines,
    index_by_government,
    mean_by_government,
    QUARTERS_PER_YEAR,
    segment_by_government,
    year_ended_growth,
)

In [2]:
# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 100

# save charts in this notebook
CHART_DIR = "./CHARTS/Political/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
CACHE_DIR = "./CACHE"
JUNE = 6
DECEMBER = 12

# ABS sources
MODELLERS_CAT, MODELLERS_TABLE = "1364.0.15.003", "1364015003"
KEY_AGGREGATES = "5206001_Key_Aggregates"

# ABS Historical Population (3105.0.65.001), data cube HPDC1. Table 2 holds the
# 30 June estimates and table 1 the 31 December ones, so between them there are
# two observations a year. Both are laid out wide: row 4 carries the years,
# column 0 the sex and column 1 the state, and the national total is footnoted
# as "Australia(e)". Table 1 reaches back to 1788, but the colonial figures are
# of no use here, so the fetch starts at 1900.
HPDC1_URL = (
    "https://www.abs.gov.au/statistics/people/population/"
    "historical-population/2021/HPDC1.xlsx"
)
HPDC1_CACHE_PREFIX = "abs_hpdc1"
HPDC1_SHEETS = {"Table 2": JUNE, "Table 1": DECEMBER}
HPDC1_HEADER_ROW = 4
HPDC1_SEX_COL, HPDC1_STATE_COL = 0, 1
HPDC1_FIRST_YEAR = 1900

# plotting constants
LFOOTER = "Australia. "
# reference line for the charts indexed to 100 at each election
INDEX_BASE_LINE = {"y": 100, "color": "darkgrey", "linewidth": 0.75}

# display charts in this notebook
SHOW = False

## Governments

Commonwealth governments since 1949, grouped into epochs of continuous
government by one side of politics.

- Each epoch starts at the **election date**, not the date the ministry was
  commissioned, because that is the date people focus on.
- The 1949 election was held on 10 December, nine days before Menzies was
  sworn in.
- The Fraser epoch starts at the 13 December 1975 election, so the month
  between the 11 November dismissal and that election sits inside Whitlam.

In [3]:
# Commonwealth governments since 1949: (election date, epoch name, party).
# An epoch is a continuous period of government by one side of politics.
GOVERNMENTS: list[tuple[str, str, str]] = [
    ("1949-12-10", "Menzies-Holt-Gorton-McMahon", "Coalition"),
    ("1972-12-02", "Whitlam", "Labor"),
    ("1975-12-13", "Fraser", "Coalition"),
    ("1983-03-05", "Hawke-Keating", "Labor"),
    ("1996-03-02", "Howard", "Coalition"),
    ("2007-11-24", "Rudd-Gillard-Rudd", "Labor"),
    ("2013-09-07", "Abbott-Turnbull-Morrison", "Coalition"),
    ("2022-05-21", "Albanese", "Labor"),
]


def get_governments() -> DataFrame:
    """Return the government epochs as a DataFrame.

    The index is the epoch name. Columns are `start` (the election date that
    began the epoch), `end` (the next epoch's election date, and today for the
    incumbent) and `party`. Each epoch is half-open: start <= date < end.
    """
    govts = DataFrame(GOVERNMENTS, columns=["start", "name", "party"])
    govts["start"] = pd.to_datetime(govts["start"])
    govts["end"] = govts["start"].shift(-1)
    govts.loc[govts.index[-1], "end"] = pd.Timestamp.today().floor("D")
    return govts.set_index("name")[["start", "end", "party"]]


governments = get_governments()
governments

,start,end,party
name,,,
Menzies-Holt-Gorton-McMahon,1949-12-10,1972-12-02,Coalition
Whitlam,1972-12-02,1975-12-13,Labor
Fraser,1975-12-13,1983-03-05,Coalition
Hawke-Keating,1983-03-05,1996-03-02,Labor
Howard,1996-03-02,2007-11-24,Coalition
Rudd-Gillard-Rudd,2007-11-24,2013-09-07,Labor
Abbott-Turnbull-Morrison,2013-09-07,2022-05-21,Coalition
Albanese,2022-05-21,2026-08-26,Labor


## Get data from the ABS

In [4]:
# Long-run quarterly headline CPI index, reconstructed back to 1948Q4 from the
# reported quarterly change (the published index is too coarsely rounded in the
# early years to compute growth from). One quarter before the 1949 election.
cpi, cpi_units, cpi_stype = get_cpi("headline")

## Plotting

In [5]:
def plot_cpi_cagr(series: Series, govts: DataFrame, stype: str) -> Series:
    """Plot compound annual inflation for each government.

    Multi-PM epoch names are stacked one PM per line, so the x-axis stays
    readable without rotation. The title says "compound annual" because this
    average is geometric, unlike the arithmetic average used for unemployment.

    Args:
        series: the long-run quarterly CPI index.
        govts: government epochs, as returned by get_governments().
        stype: the ABS series type, for the chart footer.

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Inflation Rate by Government: Average (compound annual)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        rfooter="ABS 6401.0",
        lfooter=LFOOTER + f"Headline CPI, {stype}. Election to election. ",
        show=SHOW,
    )
    return cagr


plot_cpi_cagr(cpi, governments, cpi_stype).round(2)

Menzies-Holt-Gorton-McMahon     4.45
Whitlam                        14.54
Fraser                         10.31
Hawke-Keating                   5.20
Howard                          2.58
Rudd-Gillard-Rudd               2.72
Abbott-Turnbull-Morrison        2.23
Albanese                        3.93
dtype: float64

In [6]:
def plot_cpi_cagr_path(series: Series, govts: DataFrame, stype: str) -> DataFrame:
    """Plot each government's inflation rate measured from its own election.

    Every line is solid: colour carries the party and position carries the
    government, so cycling line styles would only add noise.

    Args:
        series: the long-run quarterly CPI index.
        govts: government epochs, as returned by get_governments().
        stype: the ABS series type, for the chart footer.

    Returns:
        The plotted growth paths, in per cent per year.

    """
    paths = cagr_path_by_government(series, govts)
    index = paths.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"paths must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        paths,
        title="Inflation Rate by Government: Since Election",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr),
        y0=True,
        rfooter="ABS 6401.0",
        lfooter=LFOOTER + f"Headline CPI, {stype}. Compound annual rate from each election. ",
        show=SHOW,
    )
    return paths


_ = plot_cpi_cagr_path(cpi, governments, cpi_stype)

In [7]:
def plot_cpi_yoy(series: Series, govts: DataFrame, stype: str) -> DataFrame:
    """Plot annual CPI inflation, coloured by the party in power at the time.

    Year-ended growth at an election period measures the twelve months before
    it, so each incoming government's first few points describe its
    predecessor's prices.

    Args:
        series: the long-run quarterly CPI index.
        govts: government epochs, as returned by get_governments().
        stype: the ABS series type, for the chart footer.

    Returns:
        The plotted segments, in per cent per year.

    """
    segments = segment_by_government(year_ended_growth(series), govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        segments,
        title="Inflation Rate by Government",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        y0=True,
        rfooter="ABS 6401.0",
        lfooter=LFOOTER + f"Headline CPI, {stype}. Year-ended growth. ",
        show=SHOW,
    )
    return segments


_ = plot_cpi_yoy(cpi, governments, cpi_stype)

### Change in the annual inflation rate

Separates the *direction* of inflation from its *level*, and the two rankings
disagree: Fraser has the second-highest average inflation but reduced it, while
Abbott-Turnbull-Morrison have the lowest average and the second-largest rise.

Not a scorecard:

- **Both endpoints are single quarters.** The Abbott-Turnbull-Morrison figure
  comes almost entirely from the last two quarters of a nine-year term, as the
  post-COVID surge arrived.
- **The starting value belongs to the predecessor.** Year-ended inflation at an
  election measures the twelve months before it, so this is the incumbent's
  final-year inflation less the predecessor's.

In [8]:
def plot_cpi_yoy_change(series: Series, govts: DataFrame, stype: str) -> Series:
    """Plot the change in year-ended inflation across each government's term.

    Endpoints are the first and last quarters of each term, matching the
    unemployment change chart. The section notes set out why this compares
    governments rather than measuring what any one of them did.

    Args:
        series: the long-run quarterly CPI index.
        govts: government epochs, as returned by get_governments().
        stype: the ABS series type, for the chart footer.

    Returns:
        The plotted changes, in percentage points.

    """
    change = change_by_government(year_ended_growth(series), govts)
    mg.bar_plot_finalise(
        change.rename(index=lambda name: name.replace("-", "\n")),
        title="Inflation Rate by Government: Change Over Term (first vs last print)",
        ylabel="Percentage points",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter="ABS 6401.0",
        lfooter=LFOOTER + f"Headline CPI, {stype}. Year-ended rate, election to election. ",
        show=SHOW,
    )
    return change


plot_cpi_yoy_change(cpi, governments, cpi_stype).round(1)

Menzies-Holt-Gorton-McMahon   -3.9
Whitlam                        9.6
Fraser                        -3.1
Hawke-Keating                 -7.5
Howard                        -0.8
Rudd-Gillard-Rudd             -0.7
Abbott-Turnbull-Morrison       4.0
Albanese                      -2.3
dtype: float64

## Unemployment

No survey-based unemployment rate exists for the 1950s, and the survey was not
monthly until 1978. Three sources are combined, highest priority first:

| Source | Coverage | Basis |
|--------|----------|-------|
| Labour Force Survey (6202.0) | Feb 1978 on | published monthly rate, seasonally adjusted |
| Modellers' Database (1364.0.15.003) | 1959Q3 to 1978 | rate computed from SA quarterly counts, interpolated to monthly |
| RBA Occasional Paper 8, table 4.15 | 1950 to 1959 | **modelled** from CES registered unemployment |

- Only the first segment is a monthly survey. Before February 1978 the monthly
  detail is interpolated between quarterly observations, not measured.
- Pre-1959 is not observed at all. CES registrations are a different concept -
  people who signed on seeking full-time work - mapped onto the survey rate by
  a straight line fitted over 1960 to 1970, where the two overlap.
- Both sides of that fit are taken at June, per table note (b); fitting the
  June snapshot against a financial-year average is much weaker.
- Those annual June estimates are then interpolated to monthly, which adds no
  turning points of its own. Every chart using them says so in the footer.

In [9]:
# The three segments, the CES backcast and the splice all live in
# abs_spliced_series.get_unemployment_rate() - see its docstring for the
# segments, and the fit statistics below for how far the 1950s estimate is
# extrapolated beyond the range the line was fitted over.
unemployment, unemployment_units, unemployment_stype = get_unemployment_rate()
unemployment_report = get_unemployment_splice_report()
unemployment_fit = get_unemployment_backcast_stats()
DataFrame([unemployment_fit]).round(3)

,slope,intercept,r2,resid_sd,n,fitted_ces_min,fitted_ces_max,projected_ces_min,projected_ces_max
0,0.815,0.757,0.874,0.179,11.0,0.9,2.6,0.3,1.7


In [10]:
UR_SOURCE = "ABS 6202.0, 1364.0.15.003; RBA OP8"
UR_LFOOTER = (
    LFOOTER + "Seasonally adjusted. Pre-1978 quarterly data. "
    "Pre-1959 annual modelled from CES data. "
)


def plot_unemployment_average(series: Series, govts: DataFrame) -> Series:
    """Plot the average unemployment rate under each government.

    Args:
        series: the long-run monthly unemployment rate.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted averages, in per cent.

    """
    average = mean_by_government(series, govts)
    mg.bar_plot_finalise(
        average.rename(index=lambda name: name.replace("-", "\n")),
        title="Unemployment Rate by Government: Average",
        ylabel="Per cent",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        rfooter=UR_SOURCE,
        lfooter=UR_LFOOTER,
        show=SHOW,
    )
    return average


def plot_unemployment_change(series: Series, govts: DataFrame) -> Series:
    """Plot the change in the unemployment rate across each government's term.

    The title states the basis - first against last print - because the choice
    materially changes the result for a term that opens or closes on a turning
    point, and the chart footer has no room left for it.

    Args:
        series: the long-run monthly unemployment rate.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted changes, in percentage points.

    """
    change = change_by_government(series, govts)
    mg.bar_plot_finalise(
        change.rename(index=lambda name: name.replace("-", "\n")),
        title="Unemployment Rate by Government: Change Over Term (first vs last print)",
        ylabel="Percentage points",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=UR_SOURCE,
        lfooter=UR_LFOOTER,
        show=SHOW,
    )
    return change


DataFrame(
    {
        "average": plot_unemployment_average(unemployment, governments),
        "change": plot_unemployment_change(unemployment, governments),
    }
).round(2)

,average,change
Menzies-Holt-Gorton-McMahon,1.79,1.67
Whitlam,3.20,2.50
Fraser,6.10,4.71
Hawke-Keating,8.69,-1.53
Howard,6.35,-3.99
Rudd-Gillard-Rudd,5.12,1.28
Abbott-Turnbull-Morrison,5.61,-1.76
Albanese,3.96,0.52


In [11]:
def plot_unemployment_rate(series: Series, govts: DataFrame) -> DataFrame:
    """Plot the unemployment rate, coloured by the party in power at the time.

    The counterpart of the year-ended inflation chart. No transform is needed:
    unemployment is already a rate, so the series is plotted as published.

    Args:
        series: the long-run monthly unemployment rate.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted segments, in per cent.

    """
    segments = segment_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        segments,
        title="Unemployment Rate by Government",
        ylabel="Per cent",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr),
        rfooter=UR_SOURCE,
        lfooter=UR_LFOOTER,
        show=SHOW,
    )
    return segments


_ = plot_unemployment_rate(unemployment, governments)

## GDP per capita

- The ABS publishes GDP per capita seasonally adjusted only from 1973Q3 - a
  property of the per-capita series, not of GDP, which is adjusted back to
  1959Q3. Used as published it would drop Menzies entirely and start Whitlam
  three quarters late.
- Population has no meaningful seasonality, so the adjusted per-capita series
  is recovered for the full period as seasonally adjusted GDP over the implicit
  population (`abs_population.get_population("implicit")`). It runs from 1959Q3.
- Each government's line is rebased to 100 at its own election, so the chart
  shows how far living standards moved, not the level they started from.
- The first epoch is covered only from 1959Q3, about 55 per cent of its length.

In [12]:
GDP_SOURCE = "ABS 5206.0"
GDP_LFOOTER = (
    LFOOTER + "Chain volume measures, seasonally adjusted. "
    "Per capita derived using the implicit population. "
)


def get_real_gdp_per_capita() -> Series:
    """Real GDP per capita, seasonally adjusted, back to 1959Q3.

    The published seasonally adjusted per-capita series starts only 1973Q3
    while the aggregate is adjusted back to 1959Q3. Population has no
    meaningful seasonality, so dividing the adjusted aggregate by the implicit
    population recovers an adjusted per-capita series for the full period.

    Returns:
        Real GDP per capita, quarterly. Scale is arbitrary - every chart of it
        here is an index - so no units are returned.

    """
    gdp, _gdp_units = get_gdp(gdp_type="CVM", seasonal="SA")
    population, _pop_units = get_population("implicit")
    per_capita = (gdp / population).dropna()
    if per_capita.empty:
        raise ValueError("No overlap between GDP and the implicit population")
    return per_capita.rename("Real GDP per capita")


def plot_gdp_per_capita_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real GDP per capita, rebased to 100 at each government's election.

    Args:
        series: real GDP per capita, seasonally adjusted, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Real GDP per Capita by Government",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=GDP_SOURCE,
        lfooter=GDP_LFOOTER,
        show=SHOW,
    )
    return indexed


gdp_per_capita = get_real_gdp_per_capita()
_ = plot_gdp_per_capita_index(gdp_per_capita, governments)

In [13]:
def plot_gdp_per_capita_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual growth in real GDP per capita for each government.

    Measured first to last observation within the term. The series starts
    1959Q3, so the first epoch covers 13 of its 23 years and its bar is not
    comparable with the others. The title says "Growth" rather than "Compound
    Annual Growth" only because the longer form overruns the figure width.

    Args:
        series: real GDP per capita, seasonally adjusted, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Real GDP per Capita by Government: Growth (first vs last print)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=GDP_SOURCE,
        lfooter=GDP_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


plot_gdp_per_capita_cagr(gdp_per_capita, governments).round(2)

Menzies-Holt-Gorton-McMahon    2.45
Whitlam                        0.76
Fraser                         1.12
Hawke-Keating                  2.26
Howard                         2.38
Rudd-Gillard-Rudd              0.88
Abbott-Turnbull-Morrison       1.14
Albanese                      -0.15
dtype: float64

## Labour productivity

GDP per hour worked, spliced from two sources and plotted from Whitlam.

- **From 1978Q3**: the published whole-economy index (5206.0, seasonally
  adjusted). The trend version is not used - the ABS suspended it at 2019Q1.
- **Before 1978Q3**: real GDP over aggregate weekly hours worked from RBA OP8
  table 4.12, which observes August each year from 1966, with the quarters
  between those observations interpolated.
- Tested over the 1979-96 overlap, the derived measure's year-ended growth is
  unbiased against the published index - mean gap -0.01 percentage points - but
  noisy: sd 0.72, worst 1.95. That noise weighs most on Whitlam, whose term is
  only three years, so its bar carries more uncertainty than the chart shows.
- The splice is rebased. That is right here and wrong in most of this notebook:
  this is an index on a ratio scale, so the earlier segment can be rescaled onto
  the published level without distorting its growth.
- OP8 hours are an August snapshot and not seasonally adjusted, while GDP is, so
  the derived segment carries a level offset. The rebase absorbs it; growth is
  unaffected.
- Menzies is dropped: the hours data starts 1966, covering 7 of its 23 years.
- Albanese starts at the 2022Q1 COVID peak. From 2019Q4 to now the index is
  flat, not falling.
- Two index views, answering different questions. The **continuous** index is
  rebased once, at the 1972 election, and runs unbroken to the latest quarter,
  coloured by the party in power: it shows the level path, and how much of the
  whole post-1972 gain was banked by when. The **per-election** index restarts
  at 100 at every election, which compares terms like for like but hides where
  each one started.

In [14]:
PROD_SOURCE = "ABS 5206.0; RBA OP8"
PROD_LFOOTER = (
    LFOOTER + "Seasonally adjusted. Pre-1978Q3 derived from annual August hours. "
)

# the hours data starts 1966, covering 7 of the first epoch's 23 years
FIRST_PROD_GOVERNMENT = "Whitlam"


def plot_productivity_level(series: Series, govts: DataFrame) -> DataFrame:
    """Plot productivity as one continuous index, coloured by the party in power.

    The rebase happens once, at the first election plotted, so the whole period
    reads as a single path - where the level stands, not just how far it moved
    inside a term. Segments overlap at the election period, so consecutive
    colours join rather than leave a gap.

    Args:
        series: the spliced GDP per hour worked index, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch on a shared calendar axis.

    """
    segments = continuous_index_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")
    first_election = pd.Period(govts["start"].iloc[0], freq=index.freqstr)

    mg.line_plot_finalise(
        segments,
        title="Labour Productivity by Government: Continuous Index",
        ylabel=f"Index (= 100 at the {first_election} election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="auto"),
        rfooter=PROD_SOURCE,
        lfooter=PROD_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_productivity_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot productivity, rebased to 100 at each government's election.

    Args:
        series: the spliced GDP per hour worked index, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Labour Productivity by Government",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=PROD_SOURCE,
        lfooter=PROD_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_productivity_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual productivity growth for each government.

    Measured first to last observation within the term.

    Args:
        series: the spliced GDP per hour worked index, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Labour Productivity by Government: Growth (first vs last print)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=PROD_SOURCE,
        lfooter=PROD_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


# The published index, the RBA-hours-derived segment beneath it and the splice
# all live in abs_spliced_series.get_productivity_index().
productivity, productivity_units, productivity_stype = get_productivity_index()
productivity_report = get_productivity_splice_report()
prod_governments = governments.iloc[governments.index.get_loc(FIRST_PROD_GOVERNMENT) :]
_ = plot_productivity_level(productivity, prod_governments)
_ = plot_productivity_index(productivity, prod_governments)
plot_productivity_cagr(productivity, prod_governments).round(2)

Whitlam                     2.32
Fraser                      2.02
Hawke-Keating               1.35
Howard                      1.90
Rudd-Gillard-Rudd           1.47
Abbott-Turnbull-Morrison    0.96
Albanese                   -0.79
dtype: float64

## Real net national disposable income per capita

GDP per capita measures what is produced here; this measures what Australians
can actually spend. The difference is the terms of trade and the income paid
abroad on foreign-owned capital.

- Published seasonally adjusted only from 1973Q3, the same cut-off as GDP per
  capita and for the same reason, so it is rebuilt the same way: the
  seasonally adjusted aggregate over the implicit population. Runs from 1959Q3.
- Menzies is covered from 1959Q3, about 55 per cent of its term, as in the GDP
  per capita section.
- Read against GDP per capita: where this grows faster, the terms of trade were
  adding to income; where it grows slower, they were taking it away.

In [15]:
RNNDI_SOURCE = "ABS 5206.0"
# 16 words, to sit above the chart title without wrapping
RNNDI_DEFINITION = (
    "RNNDI = GDP plus the terms of trade gain, less net income paid abroad and depreciation"
)
RNNDI_LFOOTER = (
    LFOOTER + "Chain volume measures, seasonally adjusted. "
    "Per capita derived using the implicit population. "
)


def get_rnndi_per_capita() -> Series:
    """Real net national disposable income per capita, back to 1959Q3.

    The published per-capita series is seasonally adjusted only from 1973Q3
    while the aggregate goes back to 1959Q3, so the aggregate is divided by the
    implicit population - the same construction as real GDP per capita.

    Returns:
        RNNDI per capita, quarterly. Scale is arbitrary; every chart of it here
        is an index.

    """
    data, meta = ra.read_abs_cat("5206.0", single_excel_only=KEY_AGGREGATES, verbose=False)
    aggregate = ra.select_one(
        data,
        meta,
        {
            KEY_AGGREGATES: mc.table,
            "Real net national disposable income: Chain volume measures ;": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
    ).dropna()
    population, _units = get_population("implicit")
    per_capita = (aggregate / population).dropna()
    if per_capita.empty:
        raise ValueError("Real net national disposable income per capita is empty")
    return per_capita.rename("RNNDI per capita")


def plot_rnndi_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot RNNDI per capita, rebased to 100 at each government's election.

    Args:
        series: RNNDI per capita, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Real Net National Disposable Income per Capita by Government",
        lheader=RNNDI_DEFINITION,
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=RNNDI_SOURCE,
        lfooter=RNNDI_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_rnndi_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual growth in RNNDI per capita for each government.

    Measured first to last observation within the term, so the first epoch is
    measured over 13 of its 23 years.

    Args:
        series: RNNDI per capita, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Real Net National Disposable Income per Capita by Government: Growth",
        lheader=RNNDI_DEFINITION,
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=RNNDI_SOURCE,
        lfooter=RNNDI_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


rnndi = get_rnndi_per_capita()
_ = plot_rnndi_index(rnndi, governments)
DataFrame(
    {
        "RNNDI": plot_rnndi_cagr(rnndi, governments),
        "GDP": cagr_by_government(gdp_per_capita, governments),
    }
).round(2)

,RNNDI,GDP
Menzies-Holt-Gorton-McMahon,2.56,2.45
Whitlam,0.03,0.76
Fraser,0.55,1.12
Hawke-Keating,1.79,2.26
Howard,2.90,2.38
Rudd-Gillard-Rudd,1.35,0.88
Abbott-Turnbull-Morrison,1.68,1.14
Albanese,-1.19,-0.15


### Household disposable income

The same idea for the household sector alone, so the pair brackets the question:
what the country earns, and what lands in household hands.

- **Source**: 5206.0 table 20, `GROSS DISPOSABLE INCOME`, seasonally adjusted,
  deflated by the HFCE implicit price deflator and divided by the implicit
  population. From 1959Q3, as above.
- This is **after income tax and after transfers**, and excludes retained
  company profits. So tax cuts and welfare payments move it, and move the
  national measure not at all.
- The gap between the two is redistribution. Whitlam is the clearest case:
  households gained 2.05 per cent a year while national income was flat, as
  labour took share from profits. Hawke-Keating reverses it - the nation gained
  1.79, households 0.88 - which is the Accord.
- Gross, not net: no depreciation is deducted, unlike the national measure.

In [16]:
HDI_SOURCE = "ABS 5206.0"
HDI_LFOOTER = (
    LFOOTER + "Seasonally adjusted. Gross disposable income, HFCE deflated. "
    "Per capita derived using the implicit population. "
)
HOUSEHOLD_TABLE = "5206020_Household_Income"
DEFLATOR_TABLE = "5206005_Expenditure_Implicit_Price_Deflators"


def get_household_disposable_income() -> Series:
    """Real household gross disposable income per capita, from 1959Q3.

    Nominal household disposable income over the HFCE implicit price deflator,
    the deflator rebased to its latest value so the result is in current
    dollars, then divided by the implicit population - the same deflator and
    the same population as the real wages and per-capita sections.

    Gross, not net: no depreciation is deducted, unlike the national measure.

    Returns:
        Real household disposable income per capita, quarterly.

    """
    data, meta = ra.read_abs_cat("5206.0", single_excel_only=HOUSEHOLD_TABLE, verbose=False)
    nominal = ra.select_one(
        data,
        meta,
        {
            HOUSEHOLD_TABLE: mc.table,
            "GROSS DISPOSABLE INCOME ;": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
    ).dropna()

    defl_data, defl_meta = ra.read_abs_cat(
        "5206.0", single_excel_only=DEFLATOR_TABLE, verbose=False
    )
    deflator = ra.select_one(
        defl_data,
        defl_meta,
        {
            DEFLATOR_TABLE: mc.table,
            "Households ;  Final consumption expenditure ;": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
    ).dropna()
    deflator = deflator / deflator.iloc[-1]

    population, _units = get_population("implicit")
    per_capita = (nominal / deflator / population).dropna()
    if per_capita.empty:
        raise ValueError("Household disposable income per capita is empty")
    return per_capita.rename("Household disposable income per capita")


def plot_hdi_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot household disposable income per capita, rebased at each election.

    Args:
        series: household disposable income per capita, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Household Disposable Income per Capita by Government",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=HDI_SOURCE,
        lfooter=HDI_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_hdi_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual growth in household disposable income per capita.

    Measured first to last observation within the term, so the first epoch is
    measured over 13 of its 23 years.

    Args:
        series: household disposable income per capita, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Household Disposable Income per Capita by Government: Growth",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=HDI_SOURCE,
        lfooter=HDI_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


household_income = get_household_disposable_income()
_ = plot_hdi_index(household_income, governments)
DataFrame(
    {
        "household": plot_hdi_cagr(household_income, governments),
        "national": cagr_by_government(rnndi, governments),
        "GDP": cagr_by_government(gdp_per_capita, governments),
    }
).round(2)

,household,national,GDP
Menzies-Holt-Gorton-McMahon,2.21,2.56,2.45
Whitlam,2.05,0.03,0.76
Fraser,0.94,0.55,1.12
Hawke-Keating,0.88,1.79,2.26
Howard,2.83,2.90,2.38
Rudd-Gillard-Rudd,1.66,1.35,0.88
Abbott-Turnbull-Morrison,1.27,1.68,1.14
Albanese,-0.62,-1.19,-0.15


## Wage share

Compensation of employees as a share of total factor income - labour's slice of
what the economy produces, before tax and transfers.

- **Source**: ABS Modellers' Database (1364.0.15.003), quarterly from 1959Q3,
  the same table the unemployment splice draws on.
- Levels, averages and changes, not an index: it is already a share.
- Menzies is covered from 1959Q3, about 55 per cent of its term, so its change
  is measured over the back half of the era only.
- The two large moves are Whitlam (+5.9 points, to the series high of 62.6 in
  1975Q1) and Hawke-Keating (-5.9, the Accord unwinding it).
- Albanese is the third (+5.2), but from the series low of 49.0 at the 2022
  election back to 54.2 - the long-run mean is 54.5, so this is a return to
  normal rather than a push to a record.

In [17]:
WAGE_SHARE_SOURCE = "ABS 1364.0.15.003"
WAGE_SHARE_LFOOTER = (
    LFOOTER + "Compensation of employees as a share of total factor income. "
)


def get_wage_share() -> Series:
    """Labour's share of total factor income, quarterly from 1959Q3.

    Returns:
        The wage share, in per cent.

    """
    data, meta = ra.read_abs_cat(
        MODELLERS_CAT, single_excel_only=MODELLERS_TABLE, verbose=False
    )
    share = ra.select_one(
        data,
        meta,
        {
            MODELLERS_TABLE: mc.table,
            "Compensation of employees as a share of total factor income ;": mc.did,
        },
    ).dropna()
    if share.empty:
        raise ValueError("No wage share series found in the Modellers' Database")
    return share.rename("Wage share")


def plot_wage_share(series: Series, govts: DataFrame) -> DataFrame:
    """Plot the wage share over time, coloured by the party in power.

    Args:
        series: the wage share, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted segments, in per cent.

    """
    segments = segment_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        segments,
        title="Wage Share by Government",
        ylabel="Per cent of total factor income",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline={"y": series.mean(), "color": "darkgrey", "linewidth": 0.75},
        rfooter=WAGE_SHARE_SOURCE,
        lfooter=WAGE_SHARE_LFOOTER + "Grey line is the long-run mean. ",
        show=SHOW,
    )
    return segments


def plot_wage_share_average(series: Series, govts: DataFrame) -> Series:
    """Plot the average wage share under each government.

    Args:
        series: the wage share, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted averages, in per cent.

    """
    average = mean_by_government(series, govts)
    mg.bar_plot_finalise(
        average.rename(index=lambda name: name.replace("-", "\n")),
        title="Wage Share by Government: Average",
        ylabel="Per cent of total factor income",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        rfooter=WAGE_SHARE_SOURCE,
        lfooter=WAGE_SHARE_LFOOTER,
        show=SHOW,
    )
    return average


def plot_wage_share_change(series: Series, govts: DataFrame) -> Series:
    """Plot the change in the wage share across each government's term.

    Endpoints are the first and last quarters of each term, the same basis as
    the unemployment and inflation change charts.

    Args:
        series: the wage share, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted changes, in percentage points.

    """
    change = change_by_government(series, govts)
    mg.bar_plot_finalise(
        change.rename(index=lambda name: name.replace("-", "\n")),
        title="Wage Share by Government: Change (first vs last print)",
        ylabel="Percentage points",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=WAGE_SHARE_SOURCE,
        lfooter=WAGE_SHARE_LFOOTER,
        show=SHOW,
    )
    return change


wage_share = get_wage_share()
_ = plot_wage_share(wage_share, governments)
DataFrame(
    {
        "average": plot_wage_share_average(wage_share, governments),
        "change": plot_wage_share_change(wage_share, governments),
    }
).round(2)

,average,change
Menzies-Holt-Gorton-McMahon,53.06,6.1
Whitlam,58.92,5.9
Fraser,59.40,-1.1
Hawke-Keating,55.36,-5.9
Howard,54.47,-0.9
Rudd-Gillard-Rudd,52.33,-1.0
Abbott-Turnbull-Morrison,52.36,-3.8
Albanese,52.36,5.2


## Commonwealth taxation

Commonwealth tax receipts as a per cent of GDP, both nominal, from the national
accounts.

- **Source**: 5206.0 table 18, the national general government income account.
  Total tax is taxes on production and imports plus current taxes on income and
  wealth - the ABS publishes no single total, so the two are summed.
- Quarterly from 1972Q3, so the section runs from Whitlam. Menzies is excluded
  entirely.
- Four-quarter rolling sums over four-quarter rolling nominal GDP, which removes
  the seasonality in an Original series and matches how the ratio is normally
  quoted.
- Three views: the level, the change across each term in percentage points, and
  each term rebased to 100 at its own election. The last one answers a different
  question from the second - proportional rather than absolute movement, so a
  government starting from a low base shows a larger index move for the same
  number of percentage points.

In [18]:
TAX_SOURCE = "ABS 5206.0"
TAX_LFOOTER = (
    LFOOTER + "Commonwealth taxes as a per cent of nominal GDP. "
    "Four-quarter rolling sums. "
)
NAT_GOVT_TABLE = "5206018_Nat_Gen_Govt_Income_Account"
ROLLING_QUARTERS = 4
# the national general government account starts 1972Q3
FIRST_TAX_GOVERNMENT = "Whitlam"


def get_commonwealth_tax_ratio() -> Series:
    """Commonwealth tax receipts as a per cent of GDP, from 1973Q2.

    Total tax is taxes on production and imports plus current taxes on income
    and wealth; the ABS publishes no single total for the national general
    government account, so the two are summed. Numerator and denominator are
    both four-quarter rolling sums, which removes the seasonality in an
    Original series.

    Returns:
        Commonwealth tax as a per cent of nominal GDP, quarterly.

    """
    data, meta = ra.read_abs_cat("5206.0", single_excel_only=NAT_GOVT_TABLE, verbose=False)

    def component(did: str) -> Series:
        return ra.select_one(
            data, meta, {NAT_GOVT_TABLE: mc.table, did: mc.did, "Original": mc.stype}
        ).dropna()

    tax = (
        component("Taxes on production and imports ;")
        + component("Secondary income receivable - Total current taxes on income, wealth, etc. ;")
    ).dropna()
    gdp, _units = get_gdp(gdp_type="CP", seasonal="O")
    ratio = (
        tax.rolling(ROLLING_QUARTERS).sum() / gdp.rolling(ROLLING_QUARTERS).sum() * 100
    ).dropna()
    if ratio.empty:
        raise ValueError("No overlap between Commonwealth tax and nominal GDP")
    return ratio.rename("Commonwealth tax")


def plot_tax_ratio(series: Series, govts: DataFrame) -> DataFrame:
    """Plot Commonwealth tax as a per cent of GDP, coloured by party.

    Args:
        series: the tax ratio, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted segments.

    """
    segments = segment_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        segments,
        title="Commonwealth Taxation by Government",
        ylabel="Per cent of nominal GDP",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        rfooter=TAX_SOURCE,
        lfooter=TAX_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_tax_change(series: Series, govts: DataFrame) -> Series:
    """Plot the change in the tax ratio across each government's term.

    Endpoints are the first and last quarters of each term, the same basis as
    the unemployment, inflation and wage share change charts.

    Args:
        series: the tax ratio, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted changes, in percentage points.

    """
    change = change_by_government(series, govts)
    mg.bar_plot_finalise(
        change.rename(index=lambda name: name.replace("-", "\n")),
        title="Commonwealth Taxation by Government: Change (first vs last print)",
        ylabel="Percentage points of nominal GDP",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=TAX_SOURCE,
        lfooter=TAX_LFOOTER,
        show=SHOW,
    )
    return change


def plot_tax_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot the tax ratio rebased to 100 at each government's election.

    Proportional rather than absolute movement, so a government starting from a
    low base shows a larger index move for the same number of percentage
    points. Read alongside the change chart, not instead of it.

    Args:
        series: the tax ratio, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Commonwealth Taxation by Government: Indexed",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=TAX_SOURCE,
        lfooter=TAX_LFOOTER,
        show=SHOW,
    )
    return indexed


commonwealth_tax = get_commonwealth_tax_ratio()
tax_governments = governments.iloc[governments.index.get_loc(FIRST_TAX_GOVERNMENT) :]
_ = plot_tax_ratio(commonwealth_tax, tax_governments)
_ = plot_tax_index(commonwealth_tax, tax_governments)
DataFrame(
    {
        "average": mean_by_government(commonwealth_tax, tax_governments),
        "change": plot_tax_change(commonwealth_tax, tax_governments),
    }
).round(2)

,average,change
Whitlam,18.91,1.37
Fraser,20.87,2.06
Hawke-Keating,21.91,0.36
Howard,23.56,2.36
Rudd-Gillard-Rudd,21.70,-2.35
Abbott-Turnbull-Morrison,22.50,1.17
Albanese,23.10,0.23


### Commonwealth budget balance

Income less outlays, as a per cent of nominal GDP. In the national accounts this
is net saving: total gross income less income payable, final consumption
expenditure and depreciation, with net saving as the balancing item.

- **Source**: 5206.0 table 18, the same account as the tax series above, so the
  same 1972Q3 start and the same Whitlam-onward coverage.
- Four-quarter rolling sums over four-quarter rolling nominal GDP.
- Surplus positive. The series runs from +3.1 per cent in 1974Q3 to -8.0 in
  2021Q1, the COVID low.
- No indexed version: the series crosses zero, so rebasing it to 100 at each
  election is undefined.
- This is an accrual, current-account balance. It is not the underlying cash
  balance the Budget headlines, which is a cash measure and is not published in
  the national accounts.

In [19]:
BALANCE_SOURCE = "ABS 5206.0"
BALANCE_LFOOTER = LFOOTER + "Commonwealth net saving. Four-quarter rolling sums. "
BALANCE_YLABEL = "Per cent of nominal GDP (surplus positive)"


def get_commonwealth_balance() -> Series:
    """Commonwealth income less outlays, as a per cent of GDP, from 1973Q2.

    Net saving from the national general government income account: total gross
    income less income payable, final consumption expenditure and depreciation.
    Numerator and denominator are both four-quarter rolling sums.

    Returns:
        The balance as a per cent of nominal GDP, surplus positive, quarterly.

    """
    data, meta = ra.read_abs_cat("5206.0", single_excel_only=NAT_GOVT_TABLE, verbose=False)
    net_saving = ra.select_one(
        data, meta, {NAT_GOVT_TABLE: mc.table, "Net saving ;": mc.did, "Original": mc.stype}
    ).dropna()
    gdp, _units = get_gdp(gdp_type="CP", seasonal="O")
    balance = (
        net_saving.rolling(ROLLING_QUARTERS).sum() / gdp.rolling(ROLLING_QUARTERS).sum() * 100
    ).dropna()
    if balance.empty:
        raise ValueError("No overlap between Commonwealth net saving and nominal GDP")
    return balance.rename("Commonwealth balance")


def plot_balance(series: Series, govts: DataFrame) -> DataFrame:
    """Plot the Commonwealth balance over time, coloured by the party in power.

    Args:
        series: the balance, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted segments.

    """
    segments = segment_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        segments,
        title="Commonwealth Budget Balance by Government",
        ylabel=BALANCE_YLABEL,
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        y0=True,
        rfooter=BALANCE_SOURCE,
        lfooter=BALANCE_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_balance_average(series: Series, govts: DataFrame) -> Series:
    """Plot the average Commonwealth balance under each government.

    Args:
        series: the balance, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted averages, in per cent of GDP.

    """
    average = mean_by_government(series, govts)
    mg.bar_plot_finalise(
        average.rename(index=lambda name: name.replace("-", "\n")),
        title="Commonwealth Budget Balance by Government: Average",
        ylabel=BALANCE_YLABEL,
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=BALANCE_SOURCE,
        lfooter=BALANCE_LFOOTER,
        show=SHOW,
    )
    return average


def plot_balance_change(series: Series, govts: DataFrame) -> Series:
    """Plot the change in the balance across each government's term.

    Endpoints are the first and last quarters of each term, the same basis as
    the other change charts. A positive value is a term that ended in a better
    fiscal position than it began, whatever the level.

    Args:
        series: the balance, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted changes, in percentage points.

    """
    change = change_by_government(series, govts)
    mg.bar_plot_finalise(
        change.rename(index=lambda name: name.replace("-", "\n")),
        title="Commonwealth Budget Balance by Government: Change (first vs last print)",
        ylabel="Percentage points of nominal GDP",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=BALANCE_SOURCE,
        lfooter=BALANCE_LFOOTER,
        show=SHOW,
    )
    return change


# the same account as the tax series, so the same coverage from Whitlam on
commonwealth_balance = get_commonwealth_balance()
_ = plot_balance(commonwealth_balance, tax_governments)
DataFrame(
    {
        "average": plot_balance_average(commonwealth_balance, tax_governments),
        "change": plot_balance_change(commonwealth_balance, tax_governments),
    }
).round(2)

,average,change
Whitlam,1.91,-2.54
Fraser,-0.34,-0.90
Hawke-Keating,-0.80,-0.39
Howard,1.04,4.58
Rudd-Gillard-Rudd,-0.45,-3.14
Abbott-Turnbull-Morrison,-1.21,-0.26
Albanese,-0.01,-0.26


## Interest rates

The interbank overnight cash rate - what banks actually pay each other
overnight, which is what the RBA steers.

- **Source**: RBA table F1.1, series `FIRMMCRI`, monthly from May 1976.
- Used in preference to the cash rate *target* (`FIRMMCRT`), which starts only
  August 1990 and would drop Fraser and most of Hawke-Keating. Before 1990 the
  RBA had no announced target, so the interbank rate is the only continuous
  measure of the policy stance.
- Coverage starts within the Fraser term, which is covered from May 1976 -
  about 92 per cent of it. Menzies and Whitlam are excluded.
- Levels, averages and changes. No indexed version: this is already a rate, and
  it reaches 0.03 per cent in 2020, so rebasing to 100 would divide by almost
  nothing.

In [20]:
RATE_SOURCE = "RBA F1.1"
RATE_LFOOTER = LFOOTER + "Interbank overnight cash rate, monthly. From May 1976. "
RATE_TABLE = "F1.1"
RATE_SERIES = "FIRMMCRI"
# the series starts May 1976, inside the Fraser term
FIRST_RATE_GOVERNMENT = "Fraser"


def get_interbank_rate() -> Series:
    """Return the interbank overnight cash rate, monthly from May 1976.

    Used rather than the cash rate target, which starts only August 1990. The
    RBA announced no target before then, so this is the only continuous measure
    of the policy stance back to the 1970s.

    Returns:
        The rate in per cent per annum, monthly.

    """
    data, _meta = ra.read_rba_table(RATE_TABLE)
    if RATE_SERIES not in data.columns:
        raise ValueError(f"{RATE_SERIES} not found in RBA table {RATE_TABLE}")
    rate = data[RATE_SERIES].dropna()
    if rate.empty:
        raise ValueError(f"{RATE_SERIES} is empty")
    return rate.rename("Interbank overnight cash rate")


def plot_rate(series: Series, govts: DataFrame) -> DataFrame:
    """Plot the cash rate over time, coloured by the party in power.

    Args:
        series: the rate, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted segments, in per cent per annum.

    """
    segments = segment_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        segments,
        title="Interbank Overnight Cash Rate by Government",
        ylabel="Per cent per annum",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        rfooter=RATE_SOURCE,
        lfooter=RATE_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_rate_average(series: Series, govts: DataFrame) -> Series:
    """Plot the average cash rate under each government.

    Args:
        series: the rate, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted averages, in per cent per annum.

    """
    average = mean_by_government(series, govts)
    mg.bar_plot_finalise(
        average.rename(index=lambda name: name.replace("-", "\n")),
        title="Interbank Overnight Cash Rate by Government: Average",
        ylabel="Per cent per annum",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        rfooter=RATE_SOURCE,
        lfooter=RATE_LFOOTER,
        show=SHOW,
    )
    return average


def plot_rate_change(series: Series, govts: DataFrame) -> Series:
    """Plot the change in the cash rate across each government's term.

    Endpoints are the first and last months of each term, the same basis as the
    other change charts. This tracks the interest rate cycle, which is set by
    the RBA, not by the government of the day.

    Args:
        series: the rate, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted changes, in percentage points.

    """
    change = change_by_government(series, govts)
    mg.bar_plot_finalise(
        change.rename(index=lambda name: name.replace("-", "\n")),
        title="Interbank Overnight Cash Rate by Government: Change (first vs last print)",
        ylabel="Percentage points",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=RATE_SOURCE,
        lfooter=RATE_LFOOTER,
        show=SHOW,
    )
    return change


cash_rate = get_interbank_rate()
rate_governments = governments.iloc[governments.index.get_loc(FIRST_RATE_GOVERNMENT) :]
_ = plot_rate(cash_rate, rate_governments)
DataFrame(
    {
        "average": plot_rate_average(cash_rate, rate_governments),
        "change": plot_rate_change(cash_rate, rate_governments),
    }
).round(2)

,average,change
Fraser,10.998566,9.03
Hawke-Keating,11.351197,-9.22
Howard,5.434113,-0.82
Rudd-Gillard-Rudd,4.380662,-4.20
Abbott-Turnbull-Morrison,1.333714,-2.21
Albanese,3.67902,4.06


## Population

Quarterly population back to the 1949 election, from two sources joined at
1959Q3.

- **From 1959Q3**: the implicit population, GDP over GDP per capita (both
  Original, chain volume) - the same denominator used for GDP per capita above.
- **Before 1959Q3**: the ABS historical population cube (3105.0.65.001, HPDC1),
  which observes 30 June and 31 December each year, with the quarters between
  them interpolated.
- The two agree to a median 0.002 per cent over their 62-year overlap, so the
  join needs no rebasing and the implicit values are used as they stand.
- GDP per capita cannot follow: chain volume GDP starts 1959Q3, so that section
  still covers 55 per cent of the first epoch while this one now covers all
  of it.
- **June 1971 is a basis change**, carried by both series: people actually
  present in Australia before it, estimated resident population after. It adds
  about 0.06 percentage points a year to the Menzies growth rate.
- Population has no meaningful seasonality, so indexing to a single quarter at
  each election raises no base-period problems.

In [21]:
POP_SOURCE = "ABS 5206.0, 3105.0.65.001"
POP_LFOOTER = LFOOTER + "Implicit population. Pre-1959Q3 interpolated from semi-annual data. "


def get_historical_population() -> Series:
    """Quarterly population from 1900, from the ABS historical population cube.

    No quarterly population is published this far back - 3101.0 starts only
    1981Q2 - so the quarters are interpolated between the cube's two observed
    points a year, 30 June and 31 December. Checked against the true quarterly
    figures over 1959Q3-2021Q2, that interpolation is out by a median of 0.002
    per cent and never by more than 0.6.

    Returns:
        Population, quarterly, from 1900Q2.

    """
    content = get_file(
        HPDC1_URL, cache_dir=Path(CACHE_DIR), cache_prefix=HPDC1_CACHE_PREFIX
    )
    observations: list[Series] = []
    for sheet, month in HPDC1_SHEETS.items():
        raw = pd.read_excel(io.BytesIO(content), sheet_name=sheet, header=None)
        years = pd.to_numeric(raw.iloc[HPDC1_HEADER_ROW], errors="coerce")
        wanted = (raw[HPDC1_SEX_COL].astype(str).str.strip() == "Person") & (
            raw[HPDC1_STATE_COL].astype(str).str.strip().str.startswith("Australia")
        )
        if wanted.sum() != 1:
            raise ValueError(f"Expected one national row in {sheet}, found {wanted.sum()}")
        counts = pd.to_numeric(raw[wanted].iloc[0], errors="coerce")
        keep = years.notna() & counts.notna() & (years >= HPDC1_FIRST_YEAR)
        observations.append(
            Series(
                counts[keep].to_numpy(),
                index=pd.PeriodIndex(
                    [pd.Period(year=int(y), month=month, freq="Q") for y in years[keep]],
                    freq="Q",
                ),
            )
        )

    semi_annual = pd.concat(observations).sort_index()
    quarters = pd.period_range(semi_annual.index[0], semi_annual.index[-1], freq="Q")
    # pandas will only interpolate a PeriodIndex linearly, so the cubic fit is
    # done on timestamps and the period index put back afterwards
    gapped = semi_annual.reindex(quarters)
    gapped.index = quarters.to_timestamp()
    filled = gapped.interpolate(method="cubic")
    filled.index = quarters
    return filled.dropna().rename("Population")


def get_derived_population(govts: DataFrame) -> Series:
    """Return population, quarterly, from the first election in `govts`.

    The implicit population from 1959Q3 on, with the historical cube supplying
    the quarters before it. The implicit values are left exactly as they are;
    the backcast only fills what they do not reach. No rebasing is needed: the
    joining quarter grows 0.51 per cent against a 1955-65 mean of 0.54 (sd
    0.05), so there is no step to smooth.

    Args:
        govts: government epochs, as returned by get_governments().

    Returns:
        Population, quarterly.

    """
    implicit, _units = get_population("implicit")
    implicit = implicit.dropna()
    historical = get_historical_population()
    joined = pd.concat(
        [historical[historical.index < implicit.index[0]], implicit]
    ).sort_index()
    first_election = pd.Period(govts["start"].iloc[0], freq="Q")
    return joined[joined.index >= first_election].rename("Population")


def plot_population_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot population, rebased to 100 at each government's election.

    Args:
        series: the joined population series, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Population by Government",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=POP_SOURCE,
        lfooter=POP_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_population_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual population growth for each government.

    Measured first to last observation within the term. Every epoch is now
    covered in full, so unlike the GDP per capita chart the bars are all on the
    same footing - though the June 1971 basis change adds about 0.06 percentage
    points a year to the first of them.

    Args:
        series: the joined population series, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Population by Government: Growth (first vs last print)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=POP_SOURCE,
        lfooter=POP_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


population = get_derived_population(governments)
_ = plot_population_index(population, governments)
plot_population_cagr(population, governments).round(2)

Menzies-Holt-Gorton-McMahon    2.25
Whitlam                        1.37
Fraser                         1.31
Hawke-Keating                  1.31
Howard                         1.24
Rudd-Gillard-Rudd              1.75
Abbott-Turnbull-Morrison       1.31
Albanese                       1.93
dtype: float64

## Net overseas migration

Quarterly net overseas migration from 3101.0 table 310101, the series used in
the ABS Population notebook.

- NOM and the ERP denominator come from that same table in thousands, so the
  share is unit-clean.
- Coverage starts 1981Q2, so this section runs from Hawke-Keating onward. The
  three earlier governments are excluded rather than shown as fragments:
  Menzies and Whitlam have no data, Fraser would have 8 quarters of 29.
- Neither chart form used elsewhere in this notebook applies. Migration is a
  *flow*, so indexing to 100 at each election is meaningless, and it turns
  negative for six quarters from 2020Q2 - bottoming at -42,600 in 2020Q3 -
  which leaves a compound growth rate undefined across the sign change.
- The flow equivalents are used instead: the level over time, and the mean
  quarterly flow per government.
- The flow is charted in thousands. The population base has more than doubled
  since 1981, so raw counts flatter recent governments for reasons unrelated to
  migration policy - but the rate that corrects for that is not a mean. It is the
  compound annual contribution of NOM to population growth, in the section below:
  dividing the flow by the population it joins turns it into something that
  compounds, and that survives the six quarters of negative NOM which leave a
  growth rate of the flow itself undefined.

In [22]:
NOM_SOURCE = "ABS 3101.0"
NOM_LFOOTER = LFOOTER + "From 1981Q2. Additive decomposition, ARIMA-extended. "
ERP_TABLE = "310101"
FIRST_NOM_GOVERNMENT = "Hawke-Keating"
# the COVID border closure, as used for this series in the ABS Population notebook
NOM_DISCONTINUITY = (pd.Period("2020Q1", freq="Q"),)


def get_nom_and_erp() -> tuple[Series, Series]:
    """Quarterly net overseas migration and the resident population it sits in.

    Both come from 3101.0 table 310101 in thousands, so the ratio of the two
    needs no rescaling. This is the series used in the ABS Population notebook.

    Returns:
        Net overseas migration per quarter, and the estimated resident
        population, both quarterly from 1981Q2.

    """
    data, meta = ra.read_abs_cat("3101.0", single_excel_only=ERP_TABLE, verbose=False)
    selector = {ERP_TABLE: mc.table, "Original": mc.stype}
    _t, nom_id, _u = ra.find_abs_id(
        meta, selector | {"Net Overseas Migration ;  Australia ;": mc.did}, verbose=False
    )
    _t, erp_id, _u = ra.find_abs_id(
        meta,
        selector | {"Estimated Resident Population (ERP) ;  Australia ;": mc.did},
        verbose=False,
    )
    nom = data[ERP_TABLE][nom_id].dropna().rename("Net overseas migration")
    erp = data[ERP_TABLE][erp_id].dropna().rename("Estimated resident population")
    if nom.empty or erp.empty:
        raise ValueError("Missing net overseas migration or population in 310101")
    return nom, erp


def decompose_nom(series: Series) -> DataFrame:
    """Additive seasonal decomposition of net overseas migration.

    Additive rather than multiplicative: migration turns negative for six
    quarters from 2020. The ends are ARIMA-extended before smoothing and the
    trend is a 9-term Henderson moving average, decompose.py's quarterly
    default. The COVID border closure is passed as a discontinuity so the
    smoother does not run the trend through it.

    Args:
        series: net overseas migration, quarterly.

    Returns:
        The full decomposition, one column per step.

    """
    result = decompose(
        series,
        model="additive",
        arima_extend=True,
        discontinuity_list=list(NOM_DISCONTINUITY),
    )
    if result is None:
        raise ValueError("Seasonal decomposition of net overseas migration failed")
    return result


def plot_nom_line(series: Series, govts: DataFrame, *, share: bool, basis: str) -> DataFrame:
    """Plot net overseas migration over time, coloured by the party in power.

    Args:
        series: net overseas migration, in thousands or as a share.
        govts: the government epochs to plot.
        share: True if `series` is a percentage of population.
        basis: the decomposition output plotted, for the footer.

    Returns:
        The plotted segments.

    """
    segments = segment_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        segments,
        title=(
            "Net Overseas Migration by Government: Per Cent of Population"
            if share
            else "Net Overseas Migration by Government"
        ),
        ylabel="Per cent of population per quarter" if share else "'000 per quarter",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        y0=True,
        rfooter=NOM_SOURCE,
        lfooter=NOM_LFOOTER + f"{basis}. ",
        show=SHOW,
    )
    return segments


def plot_nom_mean(series: Series, govts: DataFrame, *, basis: str) -> Series:
    """Plot the mean quarterly migration flow under each government.

    A mean rather than a growth rate: migration is a flow that turns negative,
    so compounding it has no meaning. The rate version of this question - what
    NOM contributed to population growth - compounds properly once divided by
    the population, and is charted in the section below.

    Args:
        series: net overseas migration, in thousands.
        govts: the government epochs to plot.
        basis: the decomposition output averaged, for the footer.

    Returns:
        The plotted means, in thousands per quarter.

    """
    average = mean_by_government(series, govts)
    mg.bar_plot_finalise(
        average.rename(index=lambda name: name.replace("-", "\n")),
        title="Net Overseas Migration by Government: Mean Quarterly Flow",
        ylabel="'000 per quarter",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=NOM_SOURCE,
        lfooter=NOM_LFOOTER + f"{basis}. ",
        show=SHOW,
    )
    return average


nom, erp = get_nom_and_erp()
nom_decomposed = decompose_nom(nom)
nom_trend = nom_decomposed[FINAL_TREND].dropna().rename("NOM trend")
nom_sa = nom_decomposed[FINAL_SEASADJ].dropna().rename("NOM seasonally adjusted")
nom_governments = governments.iloc[governments.index.get_loc(FIRST_NOM_GOVERNMENT) :]

# lines use the trend - that is what the Henderson smoother is for; the means
# use the seasonally adjusted series, because the ARIMA extension synthesises
# the ends and the incumbent's term runs to the end of the data
TREND, SEASADJ = "Trend", "Seasonally adjusted"
_ = plot_nom_line(nom_trend, nom_governments, share=False, basis=TREND)
_ = plot_nom_line((nom_trend / erp * 100).dropna(), nom_governments, share=True, basis=TREND)
# the mean flow only: NOM as a rate is the compound annual contribution to population
# growth, charted in the section below, which compounds where a mean does not
DataFrame(
    {"'000/qtr": plot_nom_mean(nom_sa, nom_governments, basis=SEASADJ)}
).round(1)

,'000/qtr
Hawke-Keating,22.5
Howard,31.3
Rudd-Gillard-Rudd,58.6
Abbott-Turnbull-Morrison,45.5
Albanese,100.9


## Population growth: migration and natural increase

The population section charts how fast the population grew under each government. This one splits that growth into the two things that cause it - net overseas migration and natural increase - so the migration share of population growth is visible rather than implied.

- Each quarter's component is taken against the ERP of the quarter before, and those quarterly rates are compounded across the term. That runs the population forward as if only that component had accrued, which puts the components on the same multiplicative footing as the growth rate itself, so the two stack to the population growth rate for the term. Not exactly, though: the ERP is rebased at each Census and that intercensal adjustment belongs to neither component, which leaves the components 0.02 to 0.05 points a year above measured ERP growth - most of all for Howard, whose term spans two Censuses. Both are printed in the table below.
- **Compound annual rates on the chart, cumulative in the table.** The epochs run from about 4 years to 13, so a cumulative measure would rank them largely by how long they lasted - the same reason the rest of the notebook annualises. The cumulative figures are what actually accrued and are printed beside the term lengths that drive them.
- All three series come from 3101.0 table 310101 in thousands, quarterly from 1981Q2, so the section runs from Hawke-Keating on, the same cut as the migration section above.
- **The 2006Q3 dashed line is a definition change, not an event**: NOM moves to the 12/16-month rule from that quarter, which lifts measured migration without any change in actual movement. It falls inside the Howard term, so Howard's bar straddles two bases and is not strictly comparable with Hawke-Keating's.
- Attribution has limits worth stating: NOM is dominated by students and other temporary migrants, responds to visa settings with a lag, and the 2020-21 collapse under Morrison was a border closure rather than a migration setting. A term's number is not simply that government's policy.


In [23]:
POP_DECOMP_SOURCE = "ABS 3101.0"
POP_DECOMP_LFOOTER = LFOOTER + "From 1981Q2. Original series. "
NATURAL_INCREASE = "Natural increase"
MIGRATION = "Net overseas migration"
# the 12/16-month rule: NOM is measured on a new basis from this quarter on
NOM_METHOD_CHANGE = pd.Period("2006Q3", freq="Q")
NOM_METHOD_VLINE = {
    "text": "NOM 12/16-month rule",
    "loc": "bottom left",
    "color": "darkgrey",
    "linestyle": "--",
    "linewidth": 0.75,
}


def get_population_components() -> tuple[Series, Series, Series]:
    """Quarterly natural increase, net overseas migration and the population.

    All three come from 3101.0 table 310101 in thousands, on the one basis: the
    quarterly change in the ERP is natural increase plus NOM, give or take the
    intercensal adjustment, so the two components account for population growth
    exactly rather than one being a residual.

    Returns:
        Natural increase, net overseas migration and the ERP, from 1981Q2.

    """
    data, meta = ra.read_abs_cat("3101.0", single_excel_only=ERP_TABLE, verbose=False)
    selector = {ERP_TABLE: mc.table, "Original": mc.stype}
    wanted = {
        NATURAL_INCREASE: "Natural Increase ;  Australia ;",
        MIGRATION: "Net Overseas Migration ;  Australia ;",
        "Estimated resident population": "Estimated Resident Population (ERP) ;  Australia ;",
    }
    series: dict[str, Series] = {}
    for label, did in wanted.items():
        _t, series_id, _u = ra.find_abs_id(meta, selector | {did: mc.did}, verbose=False)
        found = data[ERP_TABLE][series_id].dropna()
        if found.empty:
            raise ValueError(f"No data for {label} in {ERP_TABLE}")
        series[label] = found.rename(label)
    return (
        series[NATURAL_INCREASE],
        series[MIGRATION],
        series["Estimated resident population"],
    )


def growth_contributions(
    components: dict[str, Series], erp: Series, govts: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """Each component's contribution to population growth under each government.

    A quarter's component is taken against the population it joined - the ERP of
    the quarter before - which gives the growth that component contributed in
    that quarter. Compounding those quarterly rates across the epoch runs the
    population forward as if only that component had accrued, so the components
    are on the same multiplicative footing as the population growth rate itself.
    They do not sum to it exactly: the ERP is rebased at each Census, and that
    intercensal adjustment belongs to neither component. The gap is 0.02 to 0.05
    points a year, largest for Howard, whose term spans two Censuses.

    Both views are returned. The annualised rate is the comparable one, because
    the epochs run from 4 years to 13 and a cumulative figure would rank them
    largely by how long they lasted; the cumulative figure is what actually
    accrued, and is worth reading beside the term lengths.

    Args:
        components: the components of population change, in the units of `erp`.
        erp: the estimated resident population, quarterly.
        govts: government epochs, as returned by get_governments().

    Returns:
        The compound annual contribution and the cumulative contribution, both
        in per cent, one row per epoch and one column per component.

    """
    annual: dict[str, Series] = {}
    cumulative: dict[str, Series] = {}
    for label, component in components.items():
        quarterly_rate = (component / erp.shift(1)).dropna()
        segments = segment_by_government(quarterly_rate, govts)
        annual_rates: dict[str, float] = {}
        cumulative_rates: dict[str, float] = {}
        for name in segments.columns:
            observed = segments[name].dropna()
            if observed.empty:
                raise ValueError(f"No data within the {name} epoch to decompose")
            compounded = float((1 + observed).prod())
            annual_rates[str(name)] = (
                compounded ** (QUARTERS_PER_YEAR / len(observed)) - 1
            ) * 100
            cumulative_rates[str(name)] = (compounded - 1) * 100
        annual[label] = Series(annual_rates)
        cumulative[label] = Series(cumulative_rates)
    return DataFrame(annual), DataFrame(cumulative)


def plot_growth_contributions(contributions: DataFrame, govts: DataFrame) -> DataFrame:
    """Plot the compound annual contribution of each component, stacked.

    The bars stack to the population growth rate for the term, bar the
    intercensal rebasing of the ERP, which belongs to neither component.

    Args:
        contributions: compound annual contributions, one column per component.
        govts: the government epochs plotted, for the footer.

    Returns:
        The plotted contributions.

    """
    change = NOM_METHOD_CHANGE
    mg.bar_plot_finalise(
        contributions.rename(index=lambda name: name.replace("-", "\n")),
        stacked=True,
        title="Population Growth by Government: Migration and Natural Increase",
        ylabel="Per cent per year",
        annotate=True,
        rounding=2,
        y0=True,
        legend={"loc": "best", "fontsize": "small"},
        rfooter=POP_DECOMP_SOURCE,
        lfooter=POP_DECOMP_LFOOTER
        + f"Compound annual rates. Excludes intercensal rebasing. NOM basis changes {change}. ",
        show=SHOW,
    )
    return contributions


def plot_contribution_lines(
    components: dict[str, Series], erp: Series, govts: DataFrame
) -> DataFrame:
    """Plot each component's year-ended contribution to population growth.

    A four-quarter sum against the population a year earlier: the annual growth
    the component contributed, free of the seasonality in births and in
    migration without any need to decompose. The dashed line marks the 2006Q3
    change in how NOM is measured, which lifts the migration line on definition
    alone.

    Args:
        components: the components of population change, in the units of `erp`.
        erp: the estimated resident population, quarterly.
        govts: the government epochs to plot.

    Returns:
        The plotted contributions, one column per component.

    """
    frame = DataFrame(
        {
            label: (
                component.rolling(QUARTERS_PER_YEAR).sum()
                / erp.shift(QUARTERS_PER_YEAR)
                * 100
            ).dropna()
            for label, component in components.items()
        }
    )
    start = pd.Period(govts["start"].iloc[0], freq="Q")
    frame = frame[frame.index >= start]
    index = frame.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"frame must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        frame,
        title="Contributions to Population Growth: Migration and Natural Increase",
        ylabel="Per cent per year",
        width=1.5,
        annotate=True,
        rounding=2,
        legend={"loc": "best", "fontsize": "small"},
        axvline=[
            *epoch_vlines(govts, index.freqstr, loc="top right"),
            {"x": NOM_METHOD_CHANGE.ordinal, **NOM_METHOD_VLINE},
        ],
        y0=True,
        rfooter=POP_DECOMP_SOURCE,
        lfooter=POP_DECOMP_LFOOTER + "Year-ended contributions. ",
        show=SHOW,
    )
    return frame


natural_increase, migration, population_erp = get_population_components()
pop_components = {MIGRATION: migration, NATURAL_INCREASE: natural_increase}
pop_annual, pop_cumulative = growth_contributions(
    pop_components, population_erp, nom_governments
)

_ = plot_growth_contributions(pop_annual, nom_governments)
_ = plot_contribution_lines(pop_components, population_erp, nom_governments)

# the cumulative view, read beside the term lengths that drive it
_terms = (
    segment_by_government(migration, nom_governments).notna().sum() / QUARTERS_PER_YEAR
)
DataFrame(
    {
        "Term (years)": _terms,
        "NOM (% pa)": pop_annual[MIGRATION],
        "Natural (% pa)": pop_annual[NATURAL_INCREASE],
        "Components (% pa)": pop_annual.sum(axis=1),
        "ERP (% pa)": cagr_by_government(population_erp, nom_governments),
        "NOM (% cumulative)": pop_cumulative[MIGRATION],
        "Natural (% cumulative)": pop_cumulative[NATURAL_INCREASE],
    }
).round(2)

,Term (years),NOM (% pa),Natural (% pa),Components (% pa),ERP (% pa),NOM (% cumulative),Natural (% cumulative)
Hawke-Keating,13.25,0.55,0.79,1.34,1.31,7.51,11.02
Howard,12.00,0.64,0.65,1.29,1.24,7.96,8.12
Rudd-Gillard-Rudd,6.00,1.07,0.72,1.79,1.75,6.61,4.40
Abbott-Turnbull-Morrison,9.00,0.74,0.59,1.33,1.31,6.91,5.41
Albanese,3.75,1.50,0.39,1.89,1.91,5.72,1.48


## Real wages

Average compensation per employee, deflated by the household final consumption
expenditure (HFCE) implicit price deflator - the method used in the ABS
Quarterly National Accounts 5206 notebook, with the deflator rebased to the
latest quarter so the series is in current dollars.

- The deflator is not the constraint: it runs from 1959Q3, as long as GDP.
- The employee count is. The wage *bill* goes back to 1959Q3, but the ABS
  divides it by employees only from 1971Q3 (non-farm) and 1978Q1 (all economy).
  The non-farm variant is used because the all-economy one would drop Whitlam
  entirely and cut two years off Fraser.
- Compensation per *hour* would be the better measure - it separates pay from
  the shift to part-time work - but starts only 1985Q3, costing three whole
  governments. These figures are therefore not hours-adjusted, and understate
  hourly pay growth from the 1980s onward.
- Menzies-Holt-Gorton-McMahon is dropped from every chart here: the data would
  cover 6 of its 92 quarters. Every remaining government is covered in full.
- Each deflator gets two index views, as for productivity and house prices: a
  **continuous** index rebased once at the 1972 election, which shows the level
  path, and one rebased at **each election**, which compares terms like for
  like but hides where each one started.

### The Hawke-Keating result

Real wages grew 0.02 per cent a year across the thirteen years to 1996 -
essentially where they started. Checked against outside sources, it holds:

- The World Socialist Web Site's account of the Accord gives the annual real
  increase per employee as "slightly less than zero" for Hawke-Keating, "more
  than 4 per cent" for Whitlam and "slightly less than 2 per cent" for Fraser.
  This notebook gets 0.02, 4.82 and 1.26.
- The RBA's 1996 Annual Report describes "the imbalance between real wages and
  productivity" being "redressed in the 1980s, assisted by the Accord" - the
  same fact from the policy side.
- Adept Economics cites a 12.2 per cent real increase over Clyde Cameron's
  ~2.5 years as Labour Minister, roughly 4.7 per cent a year - consistent with
  the Whitlam figure here.

Fraser is the one that does not line up (1.26 against "slightly less than 2").
The likely causes are all in construction: deflating by the CPI rather than the
HFCE deflator, using the all-economy rather than the non-farm series, or dating
his term from the November 1975 dismissal rather than the December election.

The flat result is not an artefact of the deflator or the non-farm variant. It
is the Accord working as designed - wage restraint traded for employment and
the social wage - and it sits directly against 2.3 per cent annual growth in
real GDP per capita over the same government.

In [24]:
WAGE_SOURCE = "ABS 5206.0"
WAGE_LFOOTER = (
    LFOOTER + "Non-farm, seasonally adjusted. "
    "Compensation per employee deflated by the HFCE deflator. "
)
WAGE_TABLE = "5206024_Selected_Analytical_Series"
DEFLATOR_TABLE = "5206005_Expenditure_Implicit_Price_Deflators"


def get_real_wages() -> Series:
    """Real average non-farm compensation per employee, from 1971Q3.

    Nominal compensation per employee over the HFCE implicit price deflator,
    the deflator rebased to its latest value so the result is in current
    dollars - the method used in the 5206 notebook. The non-farm variant is
    used because the all-economy series starts only 1978Q1, which would drop
    Whitlam entirely.

    Returns:
        Real compensation per employee, quarterly, in latest-quarter dollars.

    """
    wage_data, wage_meta = ra.read_abs_cat("5206.0", single_excel_only=WAGE_TABLE, verbose=False)
    _t, wage_id, _u = ra.find_abs_id(
        wage_meta,
        {
            WAGE_TABLE: mc.table,
            "Average non-farm compensation per employee: Current prices ;": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
        verbose=False,
    )
    nominal = wage_data[WAGE_TABLE][wage_id].dropna()

    defl_data, defl_meta = ra.read_abs_cat(
        "5206.0", single_excel_only=DEFLATOR_TABLE, verbose=False
    )
    _t, defl_id, _u = ra.find_abs_id(
        defl_meta,
        {
            DEFLATOR_TABLE: mc.table,
            "Households ;  Final consumption expenditure ;": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
        verbose=False,
    )
    deflator = defl_data[DEFLATOR_TABLE][defl_id].dropna()
    deflator = deflator / deflator.iloc[-1]  # rebase to the latest quarter

    real = (nominal / deflator).dropna()
    if real.empty:
        raise ValueError("No overlap between compensation per employee and the deflator")
    return real.rename("Real compensation per employee")


real_wages = get_real_wages()
print(f"Real wages: {real_wages.index[0]} to {real_wages.index[-1]}")

Real wages: 1971Q3 to 2026Q1


In [25]:
def plot_real_wages_level(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real wages as one continuous index, coloured by the party in power.

    The rebase happens once, at the first election plotted, so the whole period
    reads as a single path - where the level stands, not just how far it moved
    inside a term.

    Args:
        series: real compensation per employee, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch on a shared calendar axis.

    """
    segments = continuous_index_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")
    first_election = pd.Period(govts["start"].iloc[0], freq=index.freqstr)

    mg.line_plot_finalise(
        segments,
        title="Real Wages by Government: Continuous Index, HFCE Deflated",
        ylabel=f"Index (= 100 at the {first_election} election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="auto"),
        rfooter=WAGE_SOURCE,
        lfooter=WAGE_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_real_wages_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real wages, rebased to 100 at each government's election.

    Args:
        series: real compensation per employee, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Real Wages by Government: HFCE Deflated",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=WAGE_SOURCE,
        lfooter=WAGE_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_real_wages_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual growth in real wages for each government.

    Measured first to last observation within the term.

    Args:
        series: real compensation per employee, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Real Wages by Government: Growth, HFCE Deflated (first vs last print)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=WAGE_SOURCE,
        lfooter=WAGE_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


# the wage series starts 1971Q3, covering only 6 quarters of the first epoch
wage_governments = governments.iloc[1:]

_ = plot_real_wages_level(real_wages, wage_governments)
_ = plot_real_wages_index(real_wages, wage_governments)
plot_real_wages_cagr(real_wages, wage_governments).round(2)

Whitlam                     4.82
Fraser                      1.26
Hawke-Keating               0.02
Howard                      1.85
Rudd-Gillard-Rudd           1.04
Abbott-Turnbull-Morrison    0.72
Albanese                    0.18
dtype: float64

### Deflated by the CPI

The same two charts, with the reconstructed headline CPI in place of the HFCE
deflator.

- This version keeps one definition of "prices" across the whole notebook; the
  HFCE version above keeps faith with the 5206 National Accounts notebook.
  Neither is wrong - they weight housing and imputed consumption differently -
  and the pair shows how much the choice matters.
- The reconstructed CPI is **Original** while the wage series and the HFCE
  deflator are both seasonally adjusted, so a seasonal residue survives in the
  ratio. It shows as quarter-to-quarter sawtooth on the line chart, and can
  move the growth figures slightly, since those are measured between two single
  quarters.

In [26]:
WAGE_CPI_SOURCE = "ABS 5206.0, 6401.0"
# kept short: the mixed seasonal adjustment is documented in the notes above,
# and a longer footer overruns and collides with the source attribution
WAGE_CPI_LFOOTER = LFOOTER + "Non-farm compensation per employee. CPI deflated. "


def get_real_wages_cpi(price_index: Series) -> Series:
    """Real wages deflated by the CPI rather than the HFCE deflator.

    The nominal fetch repeats get_real_wages() rather than editing it, so the
    HFCE charts above are untouched. The CPI is rebased to its own final value,
    matching the HFCE treatment. Note the mixed adjustment: the wage series is
    seasonally adjusted and the reconstructed CPI is Original, so a seasonal
    residue survives in the ratio.

    Args:
        price_index: the reconstructed headline CPI.

    Returns:
        Real compensation per employee, quarterly, in latest-quarter dollars.

    """
    data, meta = ra.read_abs_cat("5206.0", single_excel_only=WAGE_TABLE, verbose=False)
    _table, series_id, _units = ra.find_abs_id(
        meta,
        {
            WAGE_TABLE: mc.table,
            "Average non-farm compensation per employee: Current prices ;": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
        verbose=False,
    )
    nominal = data[WAGE_TABLE][series_id].dropna()

    deflator = price_index / price_index.iloc[-1]  # rebase to the latest quarter
    real = (nominal / deflator).dropna()
    if real.empty:
        raise ValueError("No overlap between compensation per employee and the CPI")
    return real.rename("Real compensation per employee (CPI deflated)")


def plot_real_wages_cpi_level(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real wages as one continuous index, coloured by the party in power.

    The rebase happens once, at the first election plotted, so the whole period
    reads as a single path - where the level stands, not just how far it moved
    inside a term.

    Args:
        series: CPI-deflated compensation per employee, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch on a shared calendar axis.

    """
    segments = continuous_index_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")
    first_election = pd.Period(govts["start"].iloc[0], freq=index.freqstr)

    mg.line_plot_finalise(
        segments,
        title="Real Wages by Government: Continuous Index, CPI Deflated",
        ylabel=f"Index (= 100 at the {first_election} election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="auto"),
        rfooter=WAGE_CPI_SOURCE,
        lfooter=WAGE_CPI_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_real_wages_cpi_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot CPI-deflated real wages, rebased to 100 at each election.

    Args:
        series: CPI-deflated compensation per employee, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Real Wages by Government: CPI Deflated",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=WAGE_CPI_SOURCE,
        lfooter=WAGE_CPI_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_real_wages_cpi_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual growth in CPI-deflated real wages.

    Args:
        series: CPI-deflated compensation per employee, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Real Wages by Government: Growth, CPI Deflated (first vs last print)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=WAGE_CPI_SOURCE,
        lfooter=WAGE_CPI_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


real_wages_cpi = get_real_wages_cpi(cpi)
_ = plot_real_wages_cpi_level(real_wages_cpi, wage_governments)
_ = plot_real_wages_cpi_index(real_wages_cpi, wage_governments)
DataFrame(
    {
        "HFCE": cagr_by_government(real_wages, wage_governments),
        "CPI": plot_real_wages_cpi_cagr(real_wages_cpi, wage_governments),
    }
).round(2)

,HFCE,CPI
Whitlam,4.82,5.26
Fraser,1.26,0.91
Hawke-Keating,0.02,-0.18
Howard,1.85,1.60
Rudd-Gillard-Rudd,1.04,0.73
Abbott-Turnbull-Morrison,0.72,0.25
Albanese,0.18,0.41


## House prices

The mean price of an Australian residential dwelling, deflated by the
reconstructed headline CPI. Quarterly back to 1970, so every government from
Whitlam on is covered in full. Two index views, as for labour productivity:
one **continuous** index, rebased once at the 1972 election and coloured by
the party in power, which shows the level path; and one rebased to 100 at
**each election**, which compares terms like for like but hides where each
one started.

- The series is spliced from four measures, highest priority first: the current
  6432.0 mean price of residential dwellings (2011Q3 on), the discontinued
  6416.0 eight-capitals RPPI (2003Q3-2021Q4), the 6416.0 established-house
  index (1986Q2-2005Q2), and the BIS index for Australia underneath it
  (1970Q1-1986Q1). Each lower segment is rebased onto the mean-value dollar
  level, so the whole series reads in dollars.
- The BIS segment is not a fourth ABS series. BIS documents its Australian
  index as the ABS all-dwellings RPPI from 2003Q3 and the ABS established-house
  index from 1986Q3 - the two series already sitting above it - and **REIA
  median dwelling prices for the state capitals before that**. So all it
  contributes is 1970Q1 to 1986Q1, and that part is a median rather than a
  quality-adjusted index: it moves with the composition of what sold as well as
  with prices. The 1986Q2 junction is a real break in method, and it falls in
  the middle of the Hawke-Keating term.
- Where the two overlap, from 1986Q2, they agree closely - year-ended growth
  correlates 0.989, with a mean absolute difference of 0.62 percentage points -
  which is what justifies the join. Over the full forty years they still drift
  apart (12.1x against 13.3x), and `ra.splice()` fits one rebase factor over
  whatever overlap it is handed. Given the lot, it puts a spurious 2.7 per cent
  fall at the junction; given a single quarter, it anchors on one noisy median
  print. The BIS segment is therefore trimmed to a year past the junction. That
  choice moves Hawke-Keating over a range of about 1.6 to 2.0 per cent a year
  and touches nothing else: every other epoch sits inside a single segment,
  where the rebase factor cancels out of a growth rate.
- The spliced level is Original, and `index_by_government()` needs a
  seasonally adjusted input: the elections fall in four different quarters, so
  rebasing an original series to a single quarter would bake that quarter's
  seasonal factor into the base. Both the nominal and the real series are
  therefore decomposed (multiplicative, ARIMA-extended) and the seasonally
  adjusted component used.
- The deflator is the CPI, not the HFCE deflator used in the first real-wages
  section, so the measure is house prices against consumer prices generally.
  There is no circularity in that: the CPI covers the purchase price of *new*
  dwellings and rents, not the price of established houses.
- The 2011Q3-onward segment is a *mean* dwelling price, pulled about by the top
  of the distribution and by compositional shift - larger and better-located
  new stock - more than the hedonic 6416.0 segments beneath it are.

Fraser is the only government under which real house prices fell. Howard's is
the standout in the other direction, and nothing here separates policy from the
interest-rate cycle: that run coincides with the long fall in mortgage rates,
and the Abbott-Turnbull-Morrison figure ends at the 2022 peak.

In [27]:
HOUSE_SOURCE = "ABS 6432.0, 6416.0, 6401.0; BIS/REIA"
HOUSE_LFOOTER = LFOOTER + "Mean dwelling price, CPI deflated, seasonally adjusted. "
FIRST_HOUSE_GOVERNMENT = "Whitlam"


def plot_house_price_level(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real house prices as one continuous index, coloured by party.

    The rebase happens once, at the first election plotted, so the whole period
    reads as a single path - where the level stands, not just how far it moved
    inside a term. plot_house_price_index() answers the other question.

    Args:
        series: real mean dwelling price, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch on a shared calendar axis.

    """
    segments = continuous_index_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")
    first_election = pd.Period(govts["start"].iloc[0], freq=index.freqstr)

    mg.line_plot_finalise(
        segments,
        title="Real House Prices by Government: Continuous Index",
        ylabel=f"Index (= 100 at the {first_election} election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="auto"),
        rfooter=HOUSE_SOURCE,
        lfooter=HOUSE_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_house_price_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real house prices, rebased to 100 at each government's election.

    Args:
        series: real mean dwelling price, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Real House Prices by Government",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=HOUSE_SOURCE,
        lfooter=HOUSE_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_house_price_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual growth in real house prices for each government.

    Measured first to last observation within the term.

    Args:
        series: real mean dwelling price, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Real House Prices by Government: Growth (first vs last print)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=HOUSE_SOURCE,
        lfooter=HOUSE_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


# The splice, the BIS extension back to 1970, the CPI deflation and the seasonal
# adjustment all live in abs_prices.get_house_price_index() - see its docstring
# for the segments and the reasoning behind each choice.
nominal_house_prices, _units, _stype = get_house_price_index(
    extend_bis=True, seasonally_adjusted=True
)
real_house_prices, house_units, house_stype = get_house_price_index(
    extend_bis=True, real=True, seasonally_adjusted=True
)
house_report = get_house_price_splice_report(extend_bis=True)
print(f"House prices: {real_house_prices.index[0]} to {real_house_prices.index[-1]} ({house_units})")

# the spliced series starts 1970Q1, two elections before the Whitlam government
house_governments = governments.iloc[governments.index.get_loc(FIRST_HOUSE_GOVERNMENT) :]
_ = plot_house_price_level(real_house_prices, house_governments)
_ = plot_house_price_index(real_house_prices, house_governments)
DataFrame(
    {
        "Nominal": cagr_by_government(nominal_house_prices, house_governments),
        "Real": plot_house_price_cagr(real_house_prices, house_governments),
    }
).round(2)

House prices: 1970Q1 to 2026Q1 ($ (2026Q2 prices))


,Nominal,Real
Whitlam,17.33,2.44
Fraser,9.10,-1.14
Hawke-Keating,7.14,1.86
Howard,9.17,6.42
Rudd-Gillard-Rudd,3.25,0.54
Abbott-Turnbull-Morrison,6.82,4.47
Albanese,5.14,1.09


## Rents

The ABS CPI rents index, deflated by the reconstructed headline CPI - the
price of renting a dwelling against consumer prices generally. Quarterly back
to 1972Q3, one quarter before the 1972 election, so every government from
Whitlam on is covered in full, matching the house-price section. The same two
index views: one **continuous** index, rebased once at the 1972 election and
coloured by the party in power, which shows the level path; and one rebased to
100 at **each election**, which compares terms like for like but hides where
each one started.

- There is no splice here. The house-price series had to be stitched from four
  measures to reach back before 1986; rents come from a single ABS series - the
  CPI Rents index (6401.0 Appendix 1a), published seasonally adjusted - that
  already spans the whole period.
- Rents are a component of the CPI, at about a 6 per cent weight, so the series
  sits inside its own deflator. Deflating rents by the all-groups CPI therefore
  measures rents against the *other* ~94 per cent of the basket; it is the
  standard real-rent concept and the small self-reference does not distort the
  picture. This is the mirror of the note in the house-price section: the CPI
  covers rents and the purchase price of *new* dwellings, but not the
  established-house prices deflated there.
- The published rents index is already seasonally adjusted, so the nominal
  series is used as fetched. The real series is decomposed (multiplicative,
  ARIMA-extended) and its seasonally adjusted component used, because dividing
  by the Original reconstructed CPI reintroduces a small seasonal residue from
  the deflator, and `index_by_government()` rebases to a single election
  quarter - the same treatment as house prices.
- The CPI series is a rent *price* index - what a standing tenancy costs, the
  whole stock repriced - not market advertised rents, which turn faster. New-
  tenancy asking rents ran well ahead of this measure through 2022-24; the CPI
  series lags them because it reprices every tenancy, not just those turning
  over.

Real rents fell under Whitlam and Fraser, were roughly flat from Hawke through
Howard, and rose most under Rudd-Gillard and again under Albanese. The
Abbott-Turnbull-Morrison term is the one clear fall in real rents since the
1980s.

In [28]:
RENT_SOURCE = "ABS 6401.0"
RENT_LFOOTER = LFOOTER + "CPI rents, CPI deflated, seasonally adjusted. "
FIRST_RENT_GOVERNMENT = "Whitlam"
RENT_TABLE = "64010Appendix1a"


def get_rents(price_index: Series) -> tuple[Series, Series]:
    """Nominal and CPI-deflated CPI rents index, both seasonally adjusted.

    The rents series is the ABS CPI Rents index (6401.0 Appendix 1a), published
    seasonally adjusted and quarterly back to 1972Q3 - one quarter before the
    1972 election, so every government from Whitlam on is covered. Unlike the
    house-price splice there is nothing to join: a single ABS series carries
    the whole period. The CPI deflator is rebased to its own final value,
    matching the real-wages and house-price sections, so the real series reads
    in latest-quarter terms.

    The published rents index is already seasonally adjusted, so the nominal
    series is used as fetched. The real series is not: dividing by the Original
    reconstructed CPI reintroduces a small seasonal residue from the deflator,
    and index_by_government() rebases to a single election quarter, so the real
    series is decomposed and its seasonally adjusted component taken - the same
    treatment as house prices.

    Args:
        price_index: the reconstructed headline CPI.

    Returns:
        The nominal rents index and the real rents index, both SA.

    """
    data, meta = ra.read_abs_cat("6401.0", single_excel_only=RENT_TABLE, verbose=False)
    _table, series_id, _units = ra.find_abs_id(
        meta,
        {
            RENT_TABLE: mc.table,
            "Index Numbers": mc.did,
            "Rents ;  Australia": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
        verbose=False,
    )
    nominal = data[RENT_TABLE][series_id].dropna().rename("CPI rents")

    deflator = price_index / price_index.iloc[-1]  # rebase to the latest quarter
    real = (nominal / deflator).dropna()
    if real.empty:
        raise ValueError("No overlap between the rents series and the CPI")

    return nominal, seasonally_adjust(real.rename("Real CPI rents"))


def plot_rent_level(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real rents as one continuous index, coloured by party.

    The rebase happens once, at the first election plotted, so the whole period
    reads as a single path - where the level stands, not just how far it moved
    inside a term. plot_rent_index() answers the other question.

    Args:
        series: real CPI rents, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch on a shared calendar axis.

    """
    segments = continuous_index_by_government(series, govts)
    index = segments.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"segments must have a PeriodIndex, got {type(index).__name__}")
    first_election = pd.Period(govts["start"].iloc[0], freq=index.freqstr)

    mg.line_plot_finalise(
        segments,
        title="Real Rents by Government: Continuous Index",
        ylabel=f"Index (= 100 at the {first_election} election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="auto"),
        rfooter=RENT_SOURCE,
        lfooter=RENT_LFOOTER,
        show=SHOW,
    )
    return segments


def plot_rent_index(series: Series, govts: DataFrame) -> DataFrame:
    """Plot real rents, rebased to 100 at each government's election.

    Args:
        series: real CPI rents, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted index, one column per epoch.

    """
    indexed = index_by_government(series, govts)
    index = indexed.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"indexed must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        indexed,
        title="Real Rents by Government",
        ylabel="Index (= 100 at each election)",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="top right"),
        axhline=INDEX_BASE_LINE,
        rfooter=RENT_SOURCE,
        lfooter=RENT_LFOOTER,
        show=SHOW,
    )
    return indexed


def plot_rent_cagr(series: Series, govts: DataFrame) -> Series:
    """Plot compound annual growth in real rents for each government.

    Measured first to last observation within the term.

    Args:
        series: real CPI rents, on a PeriodIndex.
        govts: the government epochs to plot.

    Returns:
        The plotted growth rates, in per cent per year.

    """
    cagr = cagr_by_government(series, govts)
    mg.bar_plot_finalise(
        cagr.rename(index=lambda name: name.replace("-", "\n")),
        title="Real Rents by Government: Growth (first vs last print)",
        ylabel="Per cent per year",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=RENT_SOURCE,
        lfooter=RENT_LFOOTER + "Compound annual growth rate. ",
        show=SHOW,
    )
    return cagr


nominal_rents, real_rents = get_rents(cpi)
print(f"Rents: {real_rents.index[0]} to {real_rents.index[-1]}")

# the CPI rents series starts 1972Q3, the quarter before the Whitlam election
rent_governments = governments.iloc[governments.index.get_loc(FIRST_RENT_GOVERNMENT) :]
_ = plot_rent_level(real_rents, rent_governments)
_ = plot_rent_index(real_rents, rent_governments)
DataFrame(
    {
        "Nominal": cagr_by_government(nominal_rents, rent_governments),
        "Real": plot_rent_cagr(real_rents, rent_governments),
    }
).round(2)

Rents: 1972Q3 to 2026Q2


,Nominal,Real
Whitlam,12.89,-1.43
Fraser,9.54,-0.72
Hawke-Keating,5.68,0.46
Howard,3.08,0.48
Rudd-Gillard-Rudd,4.96,2.21
Abbott-Turnbull-Morrison,0.74,-1.48
Albanese,5.52,1.53


## Wages against particular prices

The rents section deflates by the all-groups CPI. This one swaps the comparator: cumulative growth in **wages** over each term, less cumulative growth in one CPI expenditure class over the same term. A positive bar means wages grew faster than the item under that government, a negative one that the item outran wages.

- The gap is a difference between two series measured over the same window, so the length of the term cancels out of the comparison in a way it does not for a cumulative growth figure on its own - which is why the other sections annualise.
- The wage comparator is the wage price index (6345.0, all sectors, total hourly rates of pay excluding bonuses, seasonally adjusted) from 1997Q3, spliced over average non-farm compensation per employee (1364.0.15.003, seasonally adjusted) back to 1971Q3. The two segments are chain-linked at a single quarter, so the junction carries average earnings' own growth rather than a step. Before 1997Q3 the measure is average earnings, which moves with the composition of employment - hours, the part-time share, the occupational mix - where the WPI holds those constant: the earlier bars answer "did pay packets outrun the item", the later ones "did the price of labour outrun it".
- The six items are the quarterly CPI expenditure class indices (6401018, Original): rents, child care, tobacco, beer, wine and spirits. Rents, tobacco and beer start in 1972Q3, wine and spirits in 1980Q3 and child care in 1982Q1, so the number of comparable terms differs by item: each is charted over the epochs whose election falls at or after its own first observation, since a term measured over part of itself is not comparable with one measured whole.
- Tobacco is on a different scale from the others - excise increases, not market prices - and the alcohol classes carry their own excise history, so each item gets its own chart rather than one shared axis.

In [29]:
WPI_GAP_SOURCE = "ABS 6345.0, 6401.0, 1364.0.15.003"
WPI_TABLE = "634501"
CLASS_TABLE = "6401018"
AENA_CAT, AENA_TABLE = "1364.0.15.003", "1364015003"
WPI_GAP_ITEMS = ("Rents", "Child care", "Tobacco", "Beer", "Wine", "Spirits")
WPI_GAP_LFOOTER = LFOOTER + (
    "Wages less CPI class, over the term. Pre-1997Q3: average earnings. "
)


def get_wpi() -> Series:
    """The WPI: total hourly rates of pay excluding bonuses, all sectors, SA.

    Quarterly from 1997Q3, so on its own it begins inside the Howard term.
    """
    data, meta = ra.read_abs_cat("6345.0", single_excel_only=WPI_TABLE, verbose=False)
    _table, series_id, _units = ra.find_abs_id(
        meta,
        {
            WPI_TABLE: mc.table,
            "Quarterly Index ;": mc.did,
            "Total hourly rates of pay excluding bonuses ;  Australia ;  Private and Public": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
        verbose=False,
    )
    return data[WPI_TABLE][series_id].dropna().rename("WPI")


def get_average_earnings() -> Series:
    """Average non-farm compensation per employee, SA, quarterly from 1971Q3.

    The Modellers' Database wage measure, which carries the comparison back
    before the WPI. It is average earnings, not a price of labour: it moves
    with the composition of employment - hours, the part-time share, the
    occupational mix - which the WPI deliberately holds constant.
    """
    data, meta = ra.read_abs_cat(AENA_CAT, single_excel_only=AENA_TABLE, verbose=False)
    _table, series_id, _units = ra.find_abs_id(
        meta,
        {
            AENA_TABLE: mc.table,
            "Non-farm ; Average compensation per employee ;": mc.did,
            "Seasonally Adjusted": mc.stype,
        },
        verbose=False,
    )
    return data[AENA_TABLE][series_id].dropna().rename("AENA")


def get_wage_series() -> tuple[Series, DataFrame]:
    """The WPI spliced over average earnings per employee.

    The WPI takes priority wherever it exists; average earnings supply growth
    beneath it, rescaled onto the WPI's level (rebase=True - both are
    ratio-scale index-like levels). The earlier segment is trimmed to the WPI's
    first quarter, so the rebase factor is a single chain-link at the junction
    rather than a mean ratio across the whole 29-year overlap: the two series
    drift apart over that span, and averaging the ratio over it puts a spurious
    5 percentage point step into the Howard term. Linked at one quarter, the
    junction quarter's growth is exactly average earnings' own growth for that
    quarter.

    Returns:
        The spliced wage index, and the ra.splice() audit report.

    """
    wpi = get_wpi()
    earnings = get_average_earnings()
    junction = earnings[earnings.index <= wpi.index[0]]
    wage, report = ra.splice([wpi, junction], rebase=True)
    return wage.rename("Wages"), report


def governments_from(series: Series, govts: DataFrame) -> DataFrame:
    """The epochs `series` covers in full: those starting at or after its start.

    The gap is a whole-of-term comparison, so an epoch that began before the
    data start would be measured over part of its term while its neighbours are
    measured over all of theirs. Such an epoch is dropped rather than reported
    short: the price classes begin at three different dates, so the number of
    comparable terms differs from item to item.

    Args:
        series: the series to be measured, on a PeriodIndex.
        govts: government epochs, as returned by get_governments().

    Returns:
        The subset of `govts` that `series` spans from each epoch's election.

    """
    index = series.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"series must have a PeriodIndex, got {type(index).__name__}")
    starts = pd.PeriodIndex(govts["start"], freq=index.freqstr)
    return govts[starts >= index[0]]


def get_cpi_class(name: str) -> Series:
    """One CPI expenditure class index (Original) from the quarterly table.

    ABS reuses a name for both a sub-group and its sole class (Tobacco, Rents),
    which gives two series IDs carrying the same index, so the first match is
    taken rather than requiring a unique one.
    """
    data, meta = ra.read_abs_cat("6401.0", single_excel_only=CLASS_TABLE, verbose=False)
    rows = ra.search_abs_meta(
        meta,
        {CLASS_TABLE: mc.table, "Index Numbers": mc.did, f";  {name} ;  Australia ;": mc.did},
        verbose=False,
    )
    if rows.empty:
        raise ValueError(f"No CPI class series found for {name!r}")
    return data[CLASS_TABLE][rows[mc.id].iloc[0]].dropna().rename(name)


def plot_wpi_gap(item_name: str, item: Series, wage: Series, govts: DataFrame) -> Series:
    """Plot cumulative wage growth less cumulative growth in one CPI class.

    Both measured over the same term, so the length of the term cancels out of
    the difference: a positive bar means wages grew faster than the item over
    that government, a negative one that the item outran wages. This is the
    same question the real-rents charts ask of the CPI, with the wage index in
    place of the consumer price index as the comparator.

    Args:
        item_name: the CPI expenditure class being compared.
        item: that class's price index, on a PeriodIndex, trimmed to the
            common start with `wage`.
        wage: the spliced wage index, on the same start.
        govts: the government epochs to plot.

    Returns:
        The plotted gap, in percentage points, indexed by epoch name.

    """
    gap = cumulative_growth_by_government(wage, govts) - cumulative_growth_by_government(
        item, govts
    )
    mg.bar_plot_finalise(
        gap.rename(index=lambda name: name.replace("-", "\n")),
        title=f"Wages vs {item_name} by Government: Cumulative Growth Gap",
        ylabel="Percentage points over the term",
        color=mg.colorise_list(govts["party"]),
        annotate=True,
        rounding=1,
        y0=True,
        rfooter=WPI_GAP_SOURCE,
        lfooter=WPI_GAP_LFOOTER,
        lheader=f"Positive = wages outran {item_name.lower()}. ",
        show=SHOW,
    )
    return gap


def plot_wpi_gap_path(item_name: str, item: Series, wage: Series, govts: DataFrame) -> DataFrame:
    """Plot the gap between wages and one CPI class as it opens across each term.

    Both series are rebased to 100 at each election, so the difference starts at
    zero and reads in percentage points of cumulative growth: above zero wages
    have grown faster than the item since that election, below zero the item has
    outrun wages. The bar chart reports where each line ends; this shows the path
    it took, and whether the term's result was one episode or a steady drift.

    Args:
        item_name: the CPI expenditure class being compared.
        item: that class's price index, on a PeriodIndex, trimmed to the
            common start with `wage`.
        wage: the spliced wage index, on the same start.
        govts: the government epochs to plot.

    Returns:
        The plotted gap path, one column per epoch on a shared calendar axis.

    """
    gap = index_by_government(wage, govts) - index_by_government(item, govts)
    index = gap.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"gap must have a PeriodIndex, got {type(index).__name__}")

    mg.line_plot_finalise(
        gap,
        title=f"Wages vs {item_name} by Government: Cumulative Growth Gap",
        tag="path",
        ylabel="Percentage points since the election",
        color=mg.colorise_list(govts["party"]),
        style="-",
        width=1.5,
        annotate=False,
        legend=False,
        axvline=epoch_vlines(govts, index.freqstr, loc="auto"),
        y0=True,
        rfooter=WPI_GAP_SOURCE,
        lfooter=WPI_GAP_LFOOTER,
        lheader=f"Positive = wages outran {item_name.lower()}. ",
        show=SHOW,
    )
    return gap


wages, wage_splice_report = get_wage_series()
print(f"Wages: {wages.index[0]} to {wages.index[-1]}")

wpi_gaps = {"Wages growth": cumulative_growth_by_government(wages, governments_from(wages, governments))}
for _item_name in WPI_GAP_ITEMS:
    _item = get_cpi_class(_item_name)
    _start = max(_item.index[0], wages.index[0])
    _item, _wages = _item[_item.index >= _start], wages[wages.index >= _start]
    _govts = governments_from(_item, governments)
    wpi_gaps[f"{_item_name} growth"] = cumulative_growth_by_government(_item, _govts)
    wpi_gaps[f"Wages less {_item_name}"] = plot_wpi_gap(_item_name, _item, _wages, _govts)
    _ = plot_wpi_gap_path(_item_name, _item, _wages, _govts)

gap_table = DataFrame(wpi_gaps)
# the items cover different epochs, so the union of their indexes is alphabetical
gap_table.reindex([_name for _name in governments.index if _name in gap_table.index]).round(1)

Wages: 1971Q3 to 2026Q2


,Wages growth,Rents growth,Wages less Rents,Child care growth,Wages less Child care,Tobacco growth,Wages less Tobacco,Beer growth,Wages less Beer,Wine growth,Wages less Wine,Spirits growth,Wages less Spirits
Whitlam,75.3,43.2,32.1,NaN,NaN,69.4,5.9,62.3,13.0,NaN,NaN,NaN,NaN
Fraser,117.4,93.7,23.7,NaN,NaN,85.5,31.9,94.3,23.1,NaN,NaN,NaN,NaN
Hawke-Keating,88.7,105.3,-16.6,117.7,-29.0,395.5,-306.7,114.5,-25.8,86.8,1.9,131.6,-42.9
Howard,54.0,43.0,11.0,45.8,8.2,92.0,-38.0,56.5,-2.6,21.4,32.6,40.0,14.0
Rudd-Gillard-Rudd,21.8,32.4,-10.6,19.0,2.8,68.0,-46.2,19.8,1.9,8.3,13.4,31.7,-10.0
Abbott-Turnbull-Morrison,20.6,6.6,14.0,33.3,-12.7,189.7,-169.1,27.8,-7.2,1.7,18.9,18.3,2.3
Albanese,15.2,24.0,-8.8,14.1,1.0,46.2,-31.0,19.5,-4.3,4.5,10.7,20.1,-4.9


## Finished

In [30]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-08-26 07:28:18

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.16.1

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.33
pandas : 3.0.5
pathlib: 1.0.1
readabs: 0.2.6

Watermark: 2.6.0



In [31]:
print("Finished")

Finished
